In [ ]:
# Azure Search
from azure.search.documents import SearchClient
from azure.search.documents.indexes import SearchIndexClient
import azure.search.documents.indexes.models as M

from azure.core.credentials import AzureKeyCredential
from openai import AzureOpenAI
from dotenv import load_dotenv
from azure.storage.blob import BlobServiceClient

#Utilitaries
import time
import traceback
from tqdm import tqdm

# Standard
import os
import json
import hashlib

In [ ]:
# Load environment variables from .env file
load_dotenv("azure_api.env")

True

In [ ]:
# Initialize the Azure Search client
client = AzureOpenAI(

    api_key=os.getenv("AZURE_API_KEY"),

    api_version=os.getenv("AZURE_API_VERSION"),

    azure_endpoint=os.getenv("AZURE_API_BASE")

) 

In [ ]:
# Configure parameters for the Azure Search
embedding_model = os.getenv("AZURE_EMBEDDING_DEPLOYMENT") # Model name for embedding
search_endpoint = os.getenv ("AZURE_SEARCH_ENDPOINT") # Azure Search endpoint
search_api_key = os.getenv ("AZURE_SEARCH_KEY") # Azure Search API key

In [285]:
name = "pyrates-dailymed"

In [ ]:
# Initialize the Azure client
index_client = SearchIndexClient(
	endpoint = search_endpoint,
	credential = AzureKeyCredential(search_api_key)
)

In [ ]:
# Algorithm configuration for vector search
algorithm_config = M.HnswAlgorithmConfiguration(
	name = f"{name}-vector-algorithm",
	kind = M.VectorSearchAlgorithmKind.HNSW,
	parameters={
		"m": 4,
		"efConstruction": 400,
		"efSearch": 500,
		"metric": M.VectorSearchAlgorithmMetric.COSINE,
	},
)

In [ ]:
# Vector search profile configuration
vector_profil_config = M.VectorSearchProfile(
	name = f"{name}-vector-profile",
	algorithm_configuration_name = f"{name}-vector-algorithm",
)

In [ ]:
# Define the index's fields
# The fields are defined with their respective data types and properties
fields = [
    M.SearchField(name="name", type=M.SearchFieldDataType.String),
    M.SearchField(name="ingredients", type=M.SearchFieldDataType.String),
    M.SearchField(name="description", type=M.SearchFieldDataType.String),
    M.SearchField(name="chunck_id", type=M.SearchFieldDataType.String, key=True),

	M.SearchField(
		name="description_vector",
		type=M.SearchFieldDataType.Collection(M.SearchFieldDataType.Single), # Vector field
		searchable=True,
		vector_search_dimensions=1536, # Dimension of the vector
		vector_search_profile_name=f"{name}-vector-profile", # Vector profile name
	),
]

In [281]:
index_name = f"{name}-index"

In [ ]:
# Create index object with the defined fields and vector search configuration
index = M.SearchIndex(
		name = index_name,
		fields = fields,
		vector_search = M.VectorSearch(
		algorithms = [algorithm_config],
		profiles= [vector_profil_config]
	)
    )

In [ ]:
try:
    # Check if the index already exists
    existing_indexes = [i.name for i in index_client.list_indexes()]
    if index_name in existing_indexes:
        print(f"L'index {index_name} existe déjà. Suppression en cours...")
        index_client.delete_index(index_name)
        print(f"L'index {index_name} a été supprimé.")
    else:
        print(f"L'index {index_name} n'existe pas. Création en cours...")

    # Create the index
    result = index_client.create_index(index)
    print(f"L'index {index_name} a été créé avec succès.")
except Exception as e:
    print(f"Une erreur s'est produite durant la gestion de l'index : {e}")

L'index pyrates-dailymed-index n'existe pas. Création en cours...
L'index pyrates-dailymed-index a été créé avec succès.


In [ ]:
try:
	# Check if the index already exists and delete it if it does
	# If the index already exists, delete it
	if index_name in [i.name for i in index_client.list_indexes()]:
		index_client.delete_index(index_name)
		print(f"{index_name} has been deleted")

	result = index_client.create_index(index)
	print(f"{name} has been created")
 
except Exception as e:
	print(f"Une erreur s'est produite durant la création de l'index : {e}")

pyrates-dailymed-index has been deleted
pyrates-dailymed has been created


In [89]:
from azure.storage.blob import BlobServiceClient, BlobClient, ContainerClient

In [ ]:
# Connect to the Azure Blob Storage account
# Replace with your actual connection string
dailymed =  os.getenv("DAILYMED_STG")

In [ ]:
# Function for generating embeddings from text
def get_embedding(text: str):
    try:
        response = client.embeddings.create(
            input=text,
            model=embedding_model
        )
        embedding = response.data[0].embedding
        print("Embedding généré (taille:", len(embedding), ")")
        return embedding
    except Exception as e:
        print(f"Erreur embedding : {e}")
        return None

In [ ]:
# Import Azure Blob Storage library
from azure.storage.blob import BlobServiceClient

blobserviceclient = BlobServiceClient.from_connection_string(dailymed)
container_client = blobserviceclient.get_container_client("dailymed-files")
blobs = container_client.list_blobs(
)

In [310]:
# Upload des documents
search_client = SearchClient(
    endpoint=search_endpoint,
    index_name=index_name,
    credential=AzureKeyCredential(search_api_key)
)

In [ ]:
# Iterate through the blobs in the container
try:
    blob_service_client = BlobServiceClient.from_connection_string(dailymed)
    container_client = blob_service_client.get_container_client("dailymed-files")
    blobs = list(container_client.list_blobs())
    print("Nombre de blobs :", len(blobs))
except Exception as e:
    print("Erreur de connexion :", e)


Nombre de blobs : 8


In [ ]:
# Define the maximum batch size for uploading documents
MAX_BATCH_SIZE = 10
batch = [] # List to store documents before uploading

blobserviceclient = BlobServiceClient.from_connection_string(dailymed)
container_client = blobserviceclient.get_container_client("dailymed-files")
blobs = container_client.list_blobs()

# Iterate through the blobs in the container
for blob in blobs:
    try:
        blob_client = container_client.get_blob_client(blob.name)
        blob_data = blob_client.download_blob().readall()
        json_data = json.loads(blob_data)
    except Exception as e:
        print(f"Erreur avec le blob {blob.name} : {e}")
        traceback.print_exc()
        continue

    # Iterate through the JSON data and prepare documents for upload
    for data in json_data:
        name = data["name"]

        # Generate a hash MD5 for the name to use as a unique identifier
        md5_hash = hashlib.md5()
        md5_hash.update(name.encode("utf-8"))
        name_hash = md5_hash.hexdigest()

        # Join ingredients into a single string
        ingredients = ";".join(data["ingredients"])
        description_full = data["description"]

        chunks = []
        start = 0
        while start < len(description_full):
            end = start + 2000
            chunks.append(description_full[start:end])
            start = end - 500

        # Process each chunk of the description
        for idx, description in enumerate(chunks):
            description_vector = get_embedding(description)
            if description_vector is None:
                continue

            chunk_id = f"{name_hash}-{idx}"
            # Create a document for each chunk
            doc = {
                "name": name,
                "ingredients": ingredients,
                "description": description,
                "chunck_id": chunk_id,
                "description_vector": description_vector
            }

            # Append the document to the batch
            batch.append(doc)

            # Upload the batch if it reaches the maximum size
            if len(batch) >= MAX_BATCH_SIZE:
                try:
                    # Upload the batch of documents to Azure Search
                    search_client.upload_documents(documents=batch)
                    print(f"✅ Uploaded batch of {len(batch)} documents.")
                    time.sleep(0.5)
                except Exception as e:
                    print(f"❌ Erreur lors de l’upload du batch : {e}")
                    traceback.print_exc()
                batch = []

# Upload the last batch if it has any documents left
if batch:
    try:
        search_client.upload_documents(documents=batch)
        print(f"✅ Uploaded final batch of {len(batch)} documents.")
    except Exception as e:
        print(f"❌ Erreur lors de l’upload du dernier batch : {e}")
        traceback.print_exc()

Embedding généré (taille: 1536 )
Embedding généré (taille: 1536 )
Embedding généré (taille: 1536 )
Embedding généré (taille: 1536 )
Embedding généré (taille: 1536 )
Embedding généré (taille: 1536 )
Embedding généré (taille: 1536 )
Embedding généré (taille: 1536 )
Embedding généré (taille: 1536 )
Embedding généré (taille: 1536 )
✅ Uploaded batch of 10 documents.
Embedding généré (taille: 1536 )
Embedding généré (taille: 1536 )
Embedding généré (taille: 1536 )
Embedding généré (taille: 1536 )
Embedding généré (taille: 1536 )
Embedding généré (taille: 1536 )
Embedding généré (taille: 1536 )
Embedding généré (taille: 1536 )
Embedding généré (taille: 1536 )
Embedding généré (taille: 1536 )
✅ Uploaded batch of 10 documents.
Embedding généré (taille: 1536 )
Embedding généré (taille: 1536 )
Embedding généré (taille: 1536 )
Embedding généré (taille: 1536 )
Embedding généré (taille: 1536 )
Embedding généré (taille: 1536 )
Embedding généré (taille: 1536 )
Embedding généré (taille: 1536 )
Embeddin